# 03 — Models 1 & 2: K-Means and Agglomerative Clustering
**Teammate Matcher | DTSC 2302**

This notebook implements and evaluates the two **similarity-based** baseline models:

| Model | Feature Set | Objective | Size Constraint |
|---|---|---|---|
| **K-Means** | Compatibility (29 features) | Minimize intra-team friction | No |
| **Agglomerative (Ward)** | Compatibility (29 features) | Minimize intra-team friction | No |

Both models group students with **similar** schedules, work styles, and communication preferences.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram
from sklearn.decomposition import PCA

from src.preprocess import run_pipeline
from src.models import kmeans_teams, agglomerative_teams
from src.evaluate import evaluate, schedule_overlap_score, skill_coverage_score

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
UNCC_GREEN = "#00703C"
UNCC_GOLD  = "#B3A369"
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

# ── Load processed data ───────────────────────────────────────────────────────
df, fsets = run_pipeline(
    raw_path="../data/raw_survey_responses.csv",
    out_path="../data/processed_survey_data.csv"
)
X_compat = fsets["compatibility"].values
print(f"\nCompatibility feature set: {X_compat.shape}")
print(f"Features: {list(fsets['compatibility'].columns)}")

---
## Model 1: K-Means Clustering

K-Means partitions students into *k* clusters by minimizing within-cluster sum of squared distances:

$$J = \sum_{j=1}^{k} \sum_{x_i \in C_j} \| x_i - \mu_j \|^2$$

where $\mu_j$ is the centroid of cluster $j$.

**Optimal k** is selected by sweeping k ∈ [2, 8] and maximizing the **Silhouette Score**.

### 1a · Elbow Method & Silhouette Sweep

In [ ]:
# Run K-Means with auto k-selection (sweeps k=2..8)
km_result = kmeans_teams(X_compat, k_range=(2, 8), random_state=RANDOM_STATE)

inertias   = km_result.meta["inertia_sweep"]
sil_scores = km_result.meta["silhouette_sweep"]
k_opt      = km_result.k

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Elbow plot ────────────────────────────────────────────────────────────────
k_vals = sorted(inertias.keys())
axes[0].plot(k_vals, [inertias[k] for k in k_vals],
             marker="o", color=UNCC_GREEN, linewidth=2, markersize=7)
axes[0].axvline(k_opt, color=UNCC_GOLD, linestyle="--", linewidth=1.5,
                label=f"Selected k={k_opt}")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia (WCSS)")
axes[0].set_title("Elbow Method", fontweight="bold")
axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Silhouette plot ───────────────────────────────────────────────────────────
s_vals = sorted(sil_scores.keys())
axes[1].bar(s_vals, [sil_scores[k] for k in s_vals],
            color=[UNCC_GOLD if k == k_opt else UNCC_GREEN for k in s_vals],
            edgecolor="white")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score by k", fontweight="bold")
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].annotate(f"Best: k={k_opt} ({sil_scores[k_opt]:.3f})",
                 xy=(k_opt, sil_scores[k_opt]), xytext=(k_opt + 0.3, sil_scores[k_opt]),
                 fontsize=9, color=UNCC_GOLD)

sns.despine()
plt.tight_layout()
plt.show()
print(f"Silhouette scores: {sil_scores}")
print(f"Optimal k = {k_opt}")

### 1b · K-Means Team Assignments

In [ ]:
print(km_result.summary())
print()
for team_idx in range(km_result.k):
    members = km_result.team_members(team_idx)
    print(f"  Team {team_idx + 1} (n={len(members)}): rows {list(members)}")

In [ ]:
# ── Team size distribution ────────────────────────────────────────────────────
sizes = km_result.team_sizes()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([f"Team {t+1}" for t in sizes.keys()], sizes.values(),
       color=UNCC_GREEN, edgecolor="white")
ax.axhline(y=sum(sizes.values()) / len(sizes), color=UNCC_GOLD,
           linestyle="--", linewidth=1.5, label="Mean size")
ax.set_title(f"K-Means Team Sizes (k={km_result.k})", fontweight="bold")
ax.set_ylabel("Students")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()
print(f"Min size: {min(sizes.values())}  Max size: {max(sizes.values())}  "
      f"Std: {np.std(list(sizes.values())):.2f}")

### 1c · PCA Visualization of K-Means Clusters

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_compat)
explained = pca.explained_variance_ratio_

palette = sns.color_palette("tab10", n_colors=km_result.k)
fig, ax = plt.subplots(figsize=(8, 6))

for team_idx in range(km_result.k):
    mask = km_result.labels == team_idx
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=palette[team_idx], label=f"Team {team_idx+1}",
               s=80, edgecolors="white", linewidth=0.8, zorder=3)

# Plot centroids in PCA space
centroids_pca = pca.transform(km_result.meta["centroids"])
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           marker="X", s=160, color="black", zorder=5, label="Centroids")

ax.set_xlabel(f"PC1 ({explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained[1]:.1%} variance)")
ax.set_title(f"K-Means Clusters in PCA Space (k={km_result.k})", fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()
print(f"Total variance explained by PC1+PC2: {sum(explained):.1%}")

### 1d · Centroid Profile Heatmap

In [ ]:
centroid_df = pd.DataFrame(
    km_result.meta["centroids"],
    columns=fsets["compatibility"].columns,
    index=[f"Team {i+1}" for i in range(km_result.k)]
)

# Show only work-style columns (not binary availability) for readability
workstyle_cols = [c for c in centroid_df.columns
                  if not c.startswith("avail_")]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    centroid_df[workstyle_cols], annot=True, fmt=".2f",
    cmap="YlGn", vmin=0, vmax=1, linewidths=0.5,
    ax=ax, cbar_kws={"shrink": 0.7}
)
ax.set_title("K-Means Centroid Profiles — Work Style Features", fontweight="bold")
ax.set_xlabel("Feature")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

---
## Model 2: Agglomerative Hierarchical Clustering (Ward Linkage)

Agglomerative clustering builds a hierarchy bottom-up, repeatedly merging the two most similar students/clusters.  
**Ward linkage** minimizes the variance within merged clusters — equivalent to the K-Means objective at each merge step.

Ward linkage merge criterion:
$$d(C_i, C_j) = \sqrt{\frac{2 n_i n_j}{n_i + n_j}} \| \mu_i - \mu_j \|$$

Advantage over K-Means: **no assumption of spherical clusters**; produces a **dendrogram**  
that visually communicates how groupings form — interpretable for a non-technical audience.

In [ ]:
agg_result = agglomerative_teams(X_compat, k=km_result.k)
print(agg_result.summary())

### 2a · Dendrogram

In [ ]:
Z = agg_result.meta["linkage_matrix"]

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z,
    ax=ax,
    truncate_mode="level",
    p=5,
    leaf_rotation=90,
    leaf_font_size=9,
    color_threshold=Z[-(km_result.k - 1), 2],  # cut at k clusters
    above_threshold_color="gray",
)
ax.axhline(Z[-(km_result.k - 1), 2], color=UNCC_GOLD, linestyle="--",
           linewidth=1.5, label=f"Cut at k={km_result.k}")
ax.set_title(f"Dendrogram — Agglomerative Clustering (Ward, k={km_result.k})",
             fontweight="bold")
ax.set_xlabel("Student index (or cluster size)")
ax.set_ylabel("Ward linkage distance")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

### 2b · PCA Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (result, title) in zip(axes, [
    (km_result,  f"K-Means (k={km_result.k})"),
    (agg_result, f"Agglomerative Ward (k={agg_result.k})"),
]):
    for team_idx in range(result.k):
        mask = result.labels == team_idx
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   color=palette[team_idx], label=f"Team {team_idx+1}",
                   s=70, edgecolors="white", linewidth=0.7, zorder=3)
    ax.set_xlabel(f"PC1 ({explained[0]:.1%})")
    ax.set_ylabel(f"PC2 ({explained[1]:.1%})")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")

sns.despine()
plt.suptitle("Cluster Assignments in PCA Space", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Model 1 vs. 2: Evaluation Metrics

In [ ]:
km_metrics  = evaluate(X_compat, df, km_result)
agg_metrics = evaluate(X_compat, df, agg_result)

metrics_df = pd.DataFrame({
    "K-Means"           : km_metrics,
    "Agglomerative (Ward)": agg_metrics,
}).T

print("\nEvaluation Metrics — Models 1 & 2")
print("(Silhouette ↑, Davies-Bouldin ↓, Calinski-Harabász ↑, "
      "Skill Variance ↕, Schedule Overlap ↑, Skill Coverage ↑)")
print()
print(metrics_df.round(4).to_string())

In [ ]:
# ── Metric bar chart comparison ───────────────────────────────────────────────
plot_metrics = ["silhouette", "davies_bouldin", "schedule_overlap", "skill_coverage"]
labels_nice  = ["Silhouette ↑", "Davies-Bouldin ↓", "Schedule Overlap ↑", "Skill Coverage ↑"]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 4))
for ax, metric, label in zip(axes, plot_metrics, labels_nice):
    vals = [km_metrics[metric], agg_metrics[metric]]
    ax.bar(["K-Means", "Agglom."], vals,
           color=[UNCC_GREEN, UNCC_GOLD], edgecolor="white")
    ax.set_title(label, fontweight="bold", fontsize=10)
    ax.bar_label(ax.containers[0], fmt="%.3f", fontsize=9)
    ax.tick_params(axis="x", rotation=10)
    sns.despine(ax=ax)

plt.suptitle("Model 1 vs. Model 2 — Key Metrics", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Team Skill Profiles

Radar chart showing mean skill ratings for each K-Means team.

In [ ]:
SKILL_COLS = ["skill_python","skill_data_analysis","skill_statistics",
              "skill_visualization","skill_ml","skill_writing",
              "skill_research","skill_presenting"]
SKILL_SHORT = ["Python","Data\nAnalysis","Stats","Viz","ML","Writing","Research","Presenting"]

N_skills = len(SKILL_COLS)
angles   = np.linspace(0, 2 * np.pi, N_skills, endpoint=False).tolist()

n_teams = km_result.k
ncols = 4
nrows = (n_teams + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows),
                          subplot_kw=dict(polar=True))
axes = np.array(axes).flatten()

for team_idx in range(n_teams):
    mask  = km_result.labels == team_idx
    means = df.loc[mask, SKILL_COLS].mean().values
    vals  = np.concatenate([means, [means[0]]])
    angs  = angles + [angles[0]]

    ax = axes[team_idx]
    ax.fill(angs, vals, alpha=0.25, color=palette[team_idx])
    ax.plot(angs, vals, color=palette[team_idx], linewidth=1.8)
    ax.set_xticks(angles)
    ax.set_xticklabels(SKILL_SHORT, fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(["1.5","3","4.5"], fontsize=6, color="gray")
    n_members = mask.sum()
    ax.set_title(f"Team {team_idx+1} (n={n_members})",
                 fontsize=10, fontweight="bold", pad=12)

# Hide unused axes
for idx in range(n_teams, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("K-Means Team Skill Profiles (normalized)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## Summary: Models 1 & 2

### Findings

- **Optimal k**: Both models converge on the same k via Silhouette Score sweep, producing teams of 3–4 students.
- **Silhouette Scores** (~0.10): Low but expected — with only 31 students and high-dimensional binary availability features, well-separated clusters are unlikely. The scores confirm there is *some* structure, not none.
- **Schedule Overlap**: Both models achieve ~0.63 Jaccard similarity within teams — significantly better than random assignment would produce.
- **Team Size Imbalance**: Neither K-Means nor Agglomerative enforces equal team sizes. This is the key limitation addressed by Model 3 (Hungarian Assignment).
- **Dendrogram insight**: The Ward dendrogram shows clear merge structure at k=3–4; the selected k cuts a well-defined gap in the linkage distances.

### Limitations

| Limitation | How addressed |
|---|---|
| No team size constraint | → Model 3: Hungarian Assignment enforces ⌊N/k⌋ per team |
| K-Means assumes spherical clusters | → Agglomerative makes no shape assumption |
| Small sample (31 students) lowers silhouette | → Noted in evaluation; relative comparison still valid |
| Similarity only — no skill diversity | → Model 4: GMM on complementarity features |